In [7]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time

In [9]:
# Load SA2 and SA4 boundary shapefiles
sa2 = gpd.read_file("SA2_2021_AUST_GDA2020.shp")
sa4 = gpd.read_file("SA4_2021_AUST_GDA2020.shp")

# Check dataset size
print("SA2 shape:", sa2.shape)
print("SA4 shape:", sa4.shape)

SA2 shape: (2473, 17)
SA4 shape: (108, 13)


In [10]:
# Filter SA4 zones in Greater Sydney
greater_sydney_sa4 = sa4[
    sa4["GCC_NAME21"].str.contains("Greater Sydney", case=False, na=False)
].copy()

# Display all SA4 names in Greater Sydney
greater_sydney_sa4_names = (
    greater_sydney_sa4["SA4_NAME21"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("Number of SA4 zones in Greater Sydney:", len(greater_sydney_sa4_names))
print(greater_sydney_sa4_names)

Number of SA4 zones in Greater Sydney: 15
['Central Coast', 'Sydney - Baulkham Hills and Hawkesbury', 'Sydney - Blacktown', 'Sydney - City and Inner South', 'Sydney - Eastern Suburbs', 'Sydney - Inner South West', 'Sydney - Inner West', 'Sydney - North Sydney and Hornsby', 'Sydney - Northern Beaches', 'Sydney - Outer South West', 'Sydney - Outer West and Blue Mountains', 'Sydney - Parramatta', 'Sydney - Ryde', 'Sydney - South West', 'Sydney - Sutherland']


In [11]:
# Select Blacktown SA4
selected_sa4_name = "Sydney - Blacktown"

selected_sa4 = sa4[
    sa4["SA4_NAME21"] == selected_sa4_name
].copy()

print("Selected SA4:", selected_sa4_name)
print("Number of SA4 records:", len(selected_sa4))
display(selected_sa4[["SA4_CODE21", "SA4_NAME21", "GCC_NAME21"]])

Selected SA4: Sydney - Blacktown
Number of SA4 records: 1


,SA4_CODE21,SA4_NAME21,GCC_NAME21
15,116,Sydney - Blacktown,Greater Sydney


In [12]:
# Filter SA2 regions within Sydney - Blacktown SA4
blacktown_sa2 = sa2[
    sa2["SA4_NAME21"] == selected_sa4_name
].copy()

print("Number of SA2 regions in Sydney - Blacktown SA4:", len(blacktown_sa2))
display(
    blacktown_sa2[
        ["SA2_CODE21", "SA2_NAME21", "SA4_CODE21", "SA4_NAME21"]
    ].sort_values("SA2_NAME21")
)

Number of SA2 regions in Sydney - Blacktown SA4: 24


,SA2_CODE21,SA2_NAME21,SA4_CODE21,SA4_NAME21
328,116021562,Acacia Gardens,116,Sydney - Blacktown
336,116031313,Bidwill - Hebersham - Emerton,116,Sydney - Blacktown
319,116011303,Blacktown (East) - Kings Park,116,Sydney - Blacktown
320,116011304,Blacktown (North) - Marayong,116,Sydney - Blacktown
323,116011560,Blacktown - South,116,Sydney - Blacktown
324,116011561,Blacktown - West,116,Sydney - Blacktown
321,116011306,Doonside - Woodcroft,116,Sydney - Blacktown
337,116031314,Glendenning - Dean Park,116,Sydney - Blacktown
327,116021309,Glenwood,116,Sydney - Blacktown
338,116031315,Hassall Grove - Plumpton,116,Sydney - Blacktown


In [13]:
def get_pois_in_bbox(minx, miny, maxx, maxy):
    """Get all POIs within a bounding box from NSW POI API"""
    url = 'https://maps.six.nsw.gov.au/arcgis/rest/services/public/NSW_POI/MapServer/0/query'
    
    params = {
        'where': '1=1',
        'geometry': f'{minx},{miny},{maxx},{maxy}',
        'geometryType': 'esriGeometryEnvelope',
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': '*',
        'returnGeometry': 'true',
        'f': 'json'
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if 'features' in data:
        return data['features']
    else:
        return []

print("Function defined")

Function defined


In [14]:
all_pois = []

for idx, row in blacktown_sa2.iterrows():
    sa2_name = row['SA2_NAME21']
    sa2_code = row['SA2_CODE21']
    
    # Get bounding box of this SA2
    bounds = row['geometry'].bounds
    minx, miny, maxx, maxy = bounds
    
    print(f"Fetching POIs for {sa2_name}...")
    
    pois = get_pois_in_bbox(minx, miny, maxx, maxy)
    
    for poi in pois:
        poi['attributes']['SA2_NAME'] = sa2_name
        poi['attributes']['SA2_CODE'] = sa2_code
        all_pois.append(poi['attributes'])
    
    time.sleep(0.5)  # be polite to the API

print(f"\nTotal POIs collected: {len(all_pois)}")

Fetching POIs for Blacktown (East) - Kings Park...
Fetching POIs for Blacktown (North) - Marayong...
Fetching POIs for Doonside - Woodcroft...
Fetching POIs for Lalor Park - Kings Langley...
Fetching POIs for Blacktown - South...
Fetching POIs for Blacktown - West...
Fetching POIs for Seven Hills - Prospect...
Fetching POIs for Toongabbie - West...
Fetching POIs for Glenwood...
Fetching POIs for Acacia Gardens...
Fetching POIs for Quakers Hill...
Fetching POIs for Kellyville Ridge - The Ponds...
Fetching POIs for Marsden Park - Shanes Park...
Fetching POIs for Riverstone...
Fetching POIs for Schofields (West) - Colebee...
Fetching POIs for Schofields - East...
Fetching POIs for Stanhope Gardens - Parklea...
Fetching POIs for Bidwill - Hebersham - Emerton...
Fetching POIs for Glendenning - Dean Park...
Fetching POIs for Hassall Grove - Plumpton...
Fetching POIs for Lethbridge Park - Tregear...
Fetching POIs for Mount Druitt - Whalan...
Fetching POIs for Prospect Reservoir...
Fetching PO

In [15]:
# Convert to dataframe
poi_df = pd.DataFrame(all_pois)

print("Shape:", poi_df.shape)
print("\nColumns:", poi_df.columns.tolist())
display(poi_df.head())

Shape: (2752, 21)

Columns: ['objectid', 'topoid', 'poigroup', 'poitype', 'poiname', 'poilabel', 'poilabeltype', 'poialtlabel', 'poisourcefeatureoid', 'accesscontrol', 'startdate', 'enddate', 'lastupdate', 'msoid', 'centroidid', 'shapeuuid', 'changetype', 'processstate', 'urbanity', 'SA2_NAME', 'SA2_CODE']


,objectid,topoid,poigroup,poitype,poiname,poilabel,poilabeltype,poialtlabel,poisourcefeatureoid,accesscontrol,...,enddate,lastupdate,msoid,centroidid,shapeuuid,changetype,processstate,urbanity,SA2_NAME,SA2_CODE
0,1836,500245990,3,Park,BREWONGLE WALKWAY,BREWONGLE WALKWAY,NAMED,None,61,1,...,32503680000000,1285588392535,89017,None,0b0df18d-085e-30f0-b593-ad0595c63321,I,None,U,Blacktown (East) - Kings Park,116011303
1,1837,500246012,3,Park,None,Park,GENERIC,None,61,1,...,32503680000000,1285588392535,93537,None,70934b1e-9583-3460-a0bd-8b2712535cc8,I,None,U,Blacktown (East) - Kings Park,116011303
2,1841,500246067,3,Park,None,Park,GENERIC,None,61,1,...,32503680000000,1285588392535,85402,None,66e1a476-5450-3b8a-aaab-dbd4107fec0f,I,None,U,Blacktown (East) - Kings Park,116011303
3,1842,500246078,3,Park,None,Park,GENERIC,None,61,1,...,32503680000000,1285588392535,93109,None,2a3eaa8a-3fb9-3b1c-a730-a460c71dd892,I,None,U,Blacktown (East) - Kings Park,116011303
4,1849,500246167,3,Park,None,Park,GENERIC,None,61,1,...,32503680000000,1285588392535,87938,None,b3980d91-6ab7-3b3d-bf74-0a793d2263e1,I,None,U,Blacktown (East) - Kings Park,116011303


In [16]:
import sqlite3

# Create a local database
conn = sqlite3.connect('blacktown_pois.db')

# Save the POI data to a table
poi_df.to_sql('blacktown_pois', conn, if_exists='replace', index=False)

print("Saved to database!")
print(f"Total rows saved: {len(poi_df)}")

# Verify it saved correctly
check = pd.read_sql("SELECT COUNT(*) as total FROM blacktown_pois", conn)
print(check)

conn.close()

Saved to database!
Total rows saved: 2752
   total
0   2752


In [17]:
# Deduplicate POIs
poi_df_clean = poi_df.copy()

# Remove duplicates using topoid
poi_df_clean = poi_df_clean.drop_duplicates(subset=['topoid'], keep='first')

print("Raw POIs:", len(poi_df))
print("Cleaned POIs after deduplication:", len(poi_df_clean))

Raw POIs: 2752
Cleaned POIs after deduplication: 1627


In [18]:
poi_df_clean.to_csv('blacktown_pois_final.csv', index=False)

# Save POI count per SA2
poi_count_by_sa2 = (
    poi_df_clean
    .groupby(['SA2_NAME', 'SA2_CODE'])
    .size()
    .reset_index(name='poi_count')
    .sort_values('poi_count', ascending=False)
)

poi_count_by_sa2.to_csv('blacktown_poi_count_by_sa2.csv', index=False)

display(poi_count_by_sa2)

,SA2_NAME,SA2_CODE,poi_count
2,Blacktown (East) - Kings Park,116011303,196
11,Lalor Park - Kings Langley,116011307,156
14,Mount Druitt - Whalan,116031317,139
17,Riverstone,116021630,118
1,Bidwill - Hebersham - Emerton,116031313,117
15,Prospect Reservoir,116031318,99
21,Seven Hills - Prospect,116011626,92
3,Blacktown (North) - Marayong,116011304,83
18,Rooty Hill - Minchinbury,116031319,67
6,Doonside - Woodcroft,116011306,66
